# Overfit Radar — Exploratory Data Analysis

This notebook provides a walkthrough of the data and core statistical concepts behind **Overfit Radar**:

1. Universe overview — price histories and return distributions
2. The cost problem — how assumed costs change gross returns
3. Strategy signal illustration — SMA crossover mechanics
4. The multiple-testing problem — DSR simulation
5. Feature correlations from the trained ML dataset

> **Run order:** Make sure you have run `scripts/download_sample_universe.py` first so `data/raw/` is populated.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/overfit_radar').is_dir())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

from overfit_radar.config import get_project_root, load_costs_config
from overfit_radar.costs.engine import BacktestEngine
from overfit_radar.costs.windows import run_window
from overfit_radar.strategies.base import StrategyConfig
from overfit_radar.strategies.zoo import compute_strategy_signal
from overfit_radar.stats.basic import calculate_sharpe, calculate_max_drawdown
from overfit_radar.stats.psr_dsr import compute_psr, compute_dsr

ROOT = get_project_root()
print(f'Project root: {ROOT}')
print(f'Raw data files: {len(list((ROOT / "data/raw").glob("*.parquet")))}')

---
## 1. Universe Overview — Price Histories

In [ ]:
# Load all available symbols
raw_files = sorted((ROOT / 'data/raw').glob('*.parquet'))
symbols = [p.stem.replace('_NS', '.NS') for p in raw_files]
print(f'Universe: {len(symbols)} stocks')
print(symbols)
if not raw_files:
    raise RuntimeError('Download the sample prices before running this notebook.')

# Load a few for demonstration
stocks = {}
for sym in symbols[:6]:  # first 6
    path = ROOT / f'data/raw/{sym.replace(".NS", "_NS")}.parquet'
    df = pd.read_parquet(path)
    df = df[df.index < pd.Timestamp("2022-01-01")]  # Reserved period excluded
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    stocks[sym] = df

print(f'\nSample: {list(stocks.keys())[0]}')
stocks[list(stocks.keys())[0]].tail(3)

In [ ]:
# Normalised cumulative returns (rebased to 1.0)
fig, ax = plt.subplots(figsize=(13, 5))

for sym, df in stocks.items():
    cum = (1 + df['adjusted_close'].pct_change().fillna(0)).cumprod()
    ax.plot(cum.index, cum, label=sym, linewidth=1.2)

ax.set_title('Normalised Cumulative Returns — Sample Large-Caps (development dates before 2022)', fontsize=13)
ax.set_ylabel('Cumulative Wealth (₹1 Base)')
ax.legend(ncol=3, fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## 2. Return Distributions — Fat Tails & Why Standard Sharpe Fails

In [ ]:
from scipy import stats

sym = symbols[0] if symbols else 'RELIANCE.NS'
df = stocks.get(sym, list(stocks.values())[0])
rets = df['adjusted_close'].pct_change().dropna()

skew = float(rets.skew())
kurt = float(rets.kurtosis() + 3)  # excess → total

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram vs Normal
ax = axes[0]
ax.hist(rets, bins=100, density=True, alpha=0.6, color='steelblue', label='Actual returns')
x = np.linspace(rets.min(), rets.max(), 300)
ax.plot(x, stats.norm.pdf(x, rets.mean(), rets.std()), 'r--', lw=2, label='Normal distribution')
ax.set_title(f'{sym} Daily Returns — Fat Tails vs Normal')
ax.set_xlabel('Daily Return')
ax.legend()
ax.text(0.05, 0.95, f'Skewness: {skew:.2f}\nKurtosis: {kurt:.2f}',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# QQ Plot
ax = axes[1]
(osm, osr), (slope, intercept, r) = stats.probplot(rets, dist='norm')
ax.scatter(osm, osr, alpha=0.3, s=5, color='steelblue')
ax.plot(osm, slope * np.array(osm) + intercept, 'r--', lw=2)
ax.set_title('Q-Q Plot: Returns vs Normal (heavy tails visible)')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.grid(True, alpha=0.2)

plt.suptitle('Why Standard Sharpe Ratio is Misleading: Real Returns are NOT Normal', 
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'Skewness = {skew:.3f} (negative = more crashes than rallies)')
print(f'Excess Kurtosis = {kurt-3:.3f} (positive = fat tails, more extreme events than normal)')
print(f'=> PSR corrects for both when evaluating Sharpe confidence')

---
## 3. The Cost Problem — Assumed Costs vs. Gross Returns

In [ ]:
costs_cfg = load_costs_config()

sym = list(stocks.keys())[0]
prices_df = stocks[sym]

strat_cfg = StrategyConfig(
    strategy_id='sma_15_50',
    family='sma_crossover',
    params={'fast_window': 15, 'slow_window': 50}
)
signal = compute_strategy_signal(prices_df, strat_cfg)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cost_mults = [0.5, 1.0, 2.0]
labels = ['Low assumed costs (0.5x)', 'Base assumed costs (1x)', 'High assumed costs (2x)']
colors = ['#2196F3', '#FF9800', '#F44336']

for ax, mult, label, color in zip(axes, cost_mults, labels, colors):
    engine = BacktestEngine(costs_cfg, cost_multiplier=mult)
    result = run_window(engine, prices_df, signal)
    
    cum_gross = (1 + result['gross_return']).cumprod()
    cum_net = (1 + result['net_return']).cumprod()
    benchmark_result = run_window(engine, prices_df, pd.Series(1., index=prices_df.index))
    benchmark = (1 + benchmark_result.net_return).cumprod()
    
    sr_gross = calculate_sharpe(result['gross_return'])
    sr_net = calculate_sharpe(result['net_return'])
    
    ax.plot(cum_gross.index, cum_gross, color='#4CAF50', alpha=0.6, label=f'Gross SR={sr_gross:.2f}')
    ax.plot(cum_net.index, cum_net, color=color, lw=2, label=f'Net SR={sr_net:.2f}')
    ax.plot(benchmark.index, benchmark, color='#9E9E9E', ls='--', alpha=0.5, label='Buy & Hold')
    ax.set_title(f'Cost Regime: {label}')
    ax.set_ylabel('Cumulative Wealth (₹1)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle(f'SMA 15/50 on {sym} — How Indian Statutory Costs Erode Alpha', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. The Multiple-Testing Problem — Why DSR Matters

In [ ]:
# Simulate: even if ALL strategies have ZERO true alpha,
# the BEST one looks great just by chance
np.random.seed(42)

T = 1000       # 4 years of daily bars
N_trials_list = [1, 10, 50, 100, 250, 494, 1000]
n_simulations = 200  # Small reproducible illustration; not a fitted result

max_sharpes = []
for N in N_trials_list:
    sim_max = []
    for _ in range(n_simulations):
        # N strategies, all pure noise (mean=0)
        rets = np.random.randn(T, N) * 0.01
        sharpes = rets.mean(axis=0) / rets.std(axis=0) * np.sqrt(252)
        sim_max.append(sharpes.max())
    max_sharpes.append(np.mean(sim_max))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(N_trials_list, max_sharpes, 'o-', color='#E53935', lw=2, markersize=7)
ax.axhline(0.5, ls='--', color='gray', label='"Acceptable" SR = 0.5')
ax.axhline(1.0, ls=':', color='gray', label='"Great" SR = 1.0')
for x, y in zip(N_trials_list, max_sharpes):
    ax.annotate(f'{y:.2f}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)

ax.set_xlabel('Number of Strategies Tested (N) — All With ZERO True Alpha')
ax.set_ylabel('Expected Max Sharpe Ratio (Pure Luck)')
ax.set_title('The Multiple-Testing Illusion: More Tests → Higher "Best" Sharpe by Chance Alone')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print('=> DSR adjusts the Sharpe confidence threshold by N — strategies must beat this luck baseline')

---
## 5. SHAP Feature Importance — What Predicts OOS Survival?

In [ ]:
import json

shap_path = ROOT / 'data/models_v1/shap_importance.json'
if shap_path.exists():
    importance_df = pd.read_json(shap_path)
    top15 = importance_df.head(15)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(top15['feature'][::-1], top15['mean_absolute_shap'][::-1],
                   color=plt.cm.Blues(np.linspace(0.4, 0.9, len(top15))))
    ax.set_xlabel('Mean |SHAP Value| — Impact on Log-Odds of OOS Survival')
    ax.set_title('Top 15 Predictors of Strategy Out-of-Sample Survival\n(Last development LightGBM fit, 200 sampled rows, raw margins)')
    ax.grid(True, alpha=0.2, axis='x')
    
    # Annotate top feature
    ax.text(top15['mean_absolute_shap'].iloc[0] * 0.5,
            len(top15) - 1,
            f" Top predictor: {top15['feature'].iloc[0]}",
            va='center', fontsize=9, color='white', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('Top 5 predictors of OOS survival:')
    print(top15[['feature', 'mean_absolute_shap']].head(5).to_string(index=False))
else:
    print('Run scripts/phase6_models.py first to generate SHAP data')

---
## 6. Parameter Neighbourhood — Plateau vs. Spike Illustration

In [ ]:
# Compare two parameter regimes: a "plateau" vs an isolated "spike"
sym = list(stocks.keys())[0]
prices_df = stocks[sym]
costs_cfg = load_costs_config()
engine = BacktestEngine(costs_cfg, cost_multiplier=1.0)

fast_grid = list(range(5, 55, 5))   # 5,10,15,...,50
slow_grid = list(range(20, 220, 20)) # 20,40,...,200

print(f'Computing {len(fast_grid) * len(slow_grid)} backtest combinations...')
sharpe_matrix = np.zeros((len(fast_grid), len(slow_grid)))

for i, fast in enumerate(fast_grid):
    for j, slow in enumerate(slow_grid):
        if fast >= slow:
            sharpe_matrix[i, j] = np.nan
            continue
        cfg = StrategyConfig(
            strategy_id=f'sma_{fast}_{slow}',
            family='sma_crossover',
            params={'fast_window': fast, 'slow_window': slow}
        )
        sig = compute_strategy_signal(prices_df, cfg)
        res = run_window(engine, prices_df, sig)
        sharpe_matrix[i, j] = calculate_sharpe(res['net_return'])

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(sharpe_matrix, cmap='RdYlGn', aspect='auto',
               vmin=-0.5, vmax=0.5)
plt.colorbar(im, ax=ax, label='Net Sharpe Ratio')
ax.set_xticks(range(len(slow_grid)))
ax.set_xticklabels(slow_grid)
ax.set_yticks(range(len(fast_grid)))
ax.set_yticklabels(fast_grid)
ax.set_xlabel('Slow Window (Days)')
ax.set_ylabel('Fast Window (Days)')
ax.set_title(f'SMA Crossover — Full Parameter Surface on {sym}\n'
             'Green plateau = robust signal | Red isolated peaks = data mining artifacts')
plt.tight_layout()
plt.show()

valid = sharpe_matrix[~np.isnan(sharpe_matrix)]
print(f'Best Sharpe: {valid.max():.3f} | Median: {np.median(valid):.3f} | Pct positive: {(valid > 0).mean()*100:.1f}%')

---
## Summary

This notebook illustrates returns, assumed cost levels and multiple-testing effects.
It does not establish profitable strategies or reliable survival probabilities.
Neighbourhood plateaus and spikes are diagnostics, not proof of an edge or overfitting.
The saved SHAP snapshot covers one development fit and 200 rows; `effective_n`
measures dependence among trials. It is not parameter-neighbourhood stability.
See README and the report for the actual development comparisons, uncertainty,
and the observed deterioration in Brier scores after calibration.
